# Agentic-RAG Evidence 提取调试

逐步运行每个 cell，出错时 traceback 直接显示在 cell 下方。

In [1]:
# === 初始化：自动定位项目根目录 ===
import sys, os
from pathlib import Path

# 尝试多种方式定位项目根
for guess in [
    Path.cwd(),
    Path(os.environ.get('PWD', '')), 
    Path(__file__).parent.parent if '__file__' in dir() else Path.cwd(),
    Path('/home/wrz/code/agentic-rag'),
]:
    if (guess / 'config' / 'settings.py').exists():
        PROJECT_ROOT = guess
        break
else:
    PROJECT_ROOT = Path.cwd()

print(f'Project root: {PROJECT_ROOT}')
sys.path.insert(0, str(PROJECT_ROOT))


Project root: /home/wrz/code/agentic-rag


## 1. 选择文件

In [2]:
FILE_PATH = str(PROJECT_ROOT / "data/raw/中华人民共和国劳动法_20181229.docx")
# 也可以直接指定: FILE_PATH = "/path/to/your/file.docx"
print(f"File: {FILE_PATH}")
print(f"Exists: {Path(FILE_PATH).exists()}")

File: /home/wrz/code/agentic-rag/data/raw/中华人民共和国劳动法_20181229.docx
Exists: True


## 2. 解析文档

In [3]:
from src.document_parser.mineru_parser import MinerUParser

doc_id = Path(FILE_PATH).stem
parser = MinerUParser()
result = parser.parse(FILE_PATH, doc_id)
markdown = result.get("markdown", "")
print(f"Markdown: {len(markdown)} chars")
print(markdown[:400])

Markdown: 8877 chars
中华人民共和国劳动法

（1994年7月5日第八届全国人民代表大会常务委员会第八次会议通过　根据2009年8月27日第十一届全国人民代表大会常务委员会第十次会议《关于修改部分法律的决定》第一次修正　根据2018年12月29日第十三届全国人民代表大会常务委员会第七次会议《关于修改〈中华人民共和国劳动法〉等七部法律的决定》第二次修正）

目　　录

第一章　总　　则

第二章　促进就业

第三章　劳动合同和集体合同

第四章　工作时间和休息休假

第五章　工　　资

第六章　劳动安全卫生

第七章　女职工和未成年工特殊保护

第八章　职业培训

第九章　社会保险和福利

第十章　劳动争议

第十一章　监督检查

第十二章　法律责任

第十三章　附　　则

第一章　总　　则

第一条　为了保护劳动者的合法权益，调整劳动关系，建立和维护适应社会主义市场经济的劳动制度，促进经济发展和社会进步，根据宪


## 3. Chunk 切分

In [4]:
from config.settings import settings
from src.chunker.hierarchical_chunker import HierarchicalChunker
from src.chunker.table_chunker import TableChunker
from src.chunker.context_merger import ContextMerger
from src.document_parser.table_parser import TableParser

h_chunker = HierarchicalChunker(chunk_size=settings.chunk_size, chunk_overlap=settings.chunk_overlap)
chunks = h_chunker.chunk(doc_id, markdown)
tables = TableParser.extract_tables_from_markdown(markdown)
chunks.extend(TableChunker().chunk_tables(doc_id, tables))
chunks = ContextMerger().merge(chunks)
chunks = ContextMerger().deduplicate(chunks)
chunks = [c for c in chunks if c["text"].strip()]
print(f"Chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  [{i}] {len(c['text'])} chars | {c['text'][:80]}...")

/home/wrz/code/agentic-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks: 7
  [0] 998 chars | 中华人民共和国劳动法

（1994年7月5日第八届全国人民代表大会常务委员会第八次会议通过　根据2009年8月27日第十一届全国人民代表大会常务委员会第十次会议...
  [1] 1489 chars | 第九条　国务院劳动行政部门主管全国劳动工作。

县级以上地方人民政府劳动行政部门主管本行政区域内的劳动工作。

第二章　促进就业

第十条　国家通过促进经济和社...
  [2] 1429 chars | （一）劳动者患病或者非因工负伤，医疗期满后，不能从事原工作也不能从事由用人单位另行安排的工作的；

（二）劳动者不能胜任工作，经过培训或者调整工作岗位，仍不能胜...
  [3] 1449 chars | （一）元旦；

（二）春节；

（三）国际劳动节；

（四）国庆节；

（五）法律、法规规定的其他休假节日。

第四十一条　用人单位由于生产经营需要，经与工会和...
  [4] 1422 chars | 劳动者对用人单位管理人员违章指挥、强令冒险作业，有权拒绝执行；对危害生命安全和身体健康的行为，有权提出批评、检举和控告。

第五十七条　国家建立伤亡事故和职业病...
  [5] 1500 chars | 社会保险基金经办机构和社会保险基金监督机构的设立和职能由法律规定。

任何组织和个人不得挪用社会保险基金。

第七十五条　国家鼓励用人单位根据本单位实际情况为劳...
  [6] 1250 chars | （一）克扣或者无故拖欠劳动者工资的；

（二）拒不支付劳动者延长工作时间工资报酬的；

（三）低于当地最低工资标准支付劳动者工资的；

（四）解除劳动合同后，未...


## 4. 选中一个 chunk

In [39]:
CHUNK_IDX = 5  # 改这里选不同 chunk
text = chunks[CHUNK_IDX]["text"][:4000]
print(f"=== Chunk [{CHUNK_IDX}] ({len(text)} chars) ===")
print(text)

=== Chunk [5] (1500 chars) ===
社会保险基金经办机构和社会保险基金监督机构的设立和职能由法律规定。

任何组织和个人不得挪用社会保险基金。

第七十五条　国家鼓励用人单位根据本单位实际情况为劳动者建立补充保险。

国家提倡劳动者个人进行储蓄性保险。

第七十六条　国家发展社会福利事业，兴建公共福利设施，为劳动者休息、休养和疗养提供条件。

用人单位应当创造条件，改善集体福利，提高劳动者的福利待遇。

第十章　劳动争议

第七十七条　用人单位与劳动者发生劳动争议，当事人可以依法申请调解、仲裁、提起诉讼，也可以协商解决。

调解原则适用于仲裁和诉讼程序。

第七十八条　解决劳动争议，应当根据合法、公正、及时处理的原则，依法维护劳动争议当事人的合法权益。

第七十九条　劳动争议发生后，当事人可以向本单位劳动争议调解委员会申请调解；调解不成，当事人一方要求仲裁的，可以向劳动争议仲裁委员会申请仲裁。当事人一方也可以直接向劳动争议仲裁委员会申请仲裁。对仲裁裁决不服的，可以向人民法院提起诉讼。

第八十条　在用人单位内，可以设立劳动争议调解委员会。劳动争议调解委员会由职工代表、用人单位代表和工会代表组成。劳动争议调解委员会主任由工会代表担任。

劳动争议经调解达成协议的，当事人应当履行。

第八十一条　劳动争议仲裁委员会由劳动行政部门代表、同级工会代表、用人单位方面的代表组成。劳动争议仲裁委员会主任由劳动行政部门代表担任。

第八十二条　提出仲裁要求的一方应当自劳动争议发生之日起六十日内向劳动争议仲裁委员会提出书面申请。仲裁裁决一般应在收到仲裁申请的六十日内作出。对仲裁裁决无异议的，当事人必须履行。

第八十三条　劳动争议当事人对仲裁裁决不服的，可以自收到仲裁裁决书之日起十五日内向人民法院提起诉讼。一方当事人在法定期限内不起诉又不履行仲裁裁决的，另一方当事人可以申请人民法院强制执行。

第八十四条　因签订集体合同发生争议，当事人协商解决不成的，当地人民政府劳动行政部门可以组织有关各方协调处理。

因履行集体合同发生争议，当事人协商解决不成的，可以向劳动争议仲裁委员会申请仲裁；对仲裁裁决不服的，可以自收到仲裁裁决书之日起十五日内向人民法院提起诉讼。

第十一章　监督检查

第八十五条　县级以上各级人民政府劳动行政部门依法对用人单位遵守劳动法律、法规的情

## 5. 🚀 调 LLM 提取 evidence

In [40]:
from src.knowledge_graph.entity_extractor import EntityExtractor
from config.prompts import EVIDENCE_EXTRACTION_SYSTEM, EVIDENCE_EXTRACTION_EXAMPLE

extractor = EntityExtractor()
llm = extractor._get_llm()
prompt = f"""{EVIDENCE_EXTRACTION_SYSTEM}

Example:
{EVIDENCE_EXTRACTION_EXAMPLE}

Text: {text}

Response:"""

resp = llm.invoke(prompt)
raw = resp.content
print(f"=== Raw LLM response ({len(raw)} chars) ===")
print(raw)

=== Raw LLM response (2969 chars) ===
{
  "evidence_items": [
    {
      "type": "ENTITY",
      "entity_name": "社会保险基金经办机构",
      "entity_type": "Organization",
      "confidence": 0.95
    },
    {
      "type": "ENTITY",
      "entity_name": "社会保险基金监督机构",
      "entity_type": "Organization",
      "confidence": 0.95
    },
    {
      "type": "ENTITY",
      "entity_name": "社会保险基金",
      "entity_type": "Concept",
      "confidence": 0.95
    },
    {
      "type": "ENTITY",
      "entity_name": "国家",
      "entity_type": "Organization",
      "confidence": 0.95
    },
    {
      "type": "ENTITY",
      "entity_name": "用人单位",
      "entity_type": "Organization",
      "confidence": 0.90
    },
    {
      "type": "ENTITY",
      "entity_name": "劳动者",
      "entity_type": "Concept",
      "confidence": 0.90
    },
    {
      "type": "ENTITY",
      "entity_name": "劳动争议调解委员会",
      "entity_type": "Organization",
      "confidence": 0.95
    },
    {
      "type": "ENTITY",
      

## 6. 三步解析：regex → json.loads → 防御解析

In [41]:
import json, re, traceback

print("=== A) regex extract JSON ===")
match = re.search(r"\{[\s\S]*\}", raw)
json_str = match.group() if match else "{}"
print(f"Extracted {len(json_str)} chars")

print()
print("=== B) json.loads ===")
try:
    data = json.loads(json_str)
    print(f"Type: {type(data).__name__}")
    if isinstance(data, dict):
        ev = data.get("evidence_items", "MISSING")
        print(f"evidence_items type: {type(ev).__name__}")
        if isinstance(ev, list):
            print(f"  {len(ev)} items")
            for i, it in enumerate(ev[:3]):
                print(f"  [{i}] type={type(it).__name__}: {str(it)[:200]}")
        elif isinstance(ev, str):
            print(f"  STRING: {ev[:500]}")
        else:
            print(f"  {repr(ev)[:500]}")
    else:
        print(f"NOT DICT: {str(data)[:500]}")
        data = {}
except Exception as e:
    print(f"JSON ERROR: {e}")
    traceback.print_exc()
    data = {}

print()
print("=== C) _parse_evidence_items (defensive) ===")
from src.knowledge_graph.entity_extractor import _parse_evidence_items
items = _parse_evidence_items(data)
print(f"Parsed {len(items)} items")
for i, it in enumerate(items[:5]):
    print(f"  [{i}] type={it.get('type','?')} name={it.get('entity_name',it.get('subject_name','?'))}")

=== A) regex extract JSON ===
Extracted 2969 chars

=== B) json.loads ===
Type: dict
evidence_items type: list
  19 items
  [0] type=dict: {'type': 'ENTITY', 'entity_name': '社会保险基金经办机构', 'entity_type': 'Organization', 'confidence': 0.95}
  [1] type=dict: {'type': 'ENTITY', 'entity_name': '社会保险基金监督机构', 'entity_type': 'Organization', 'confidence': 0.95}
  [2] type=dict: {'type': 'ENTITY', 'entity_name': '社会保险基金', 'entity_type': 'Concept', 'confidence': 0.95}

=== C) _parse_evidence_items (defensive) ===
Parsed 19 items
  [0] type=ENTITY name=社会保险基金经办机构
  [1] type=ENTITY name=社会保险基金监督机构
  [2] type=ENTITY name=社会保险基金
  [3] type=ENTITY name=国家
  [4] type=ENTITY name=用人单位


## 7. 构造 Evidence 对象

In [42]:
from src.knowledge_graph.evidence import (
    EntityEvidence, EntityAttributeEvidence, FactEvidence, FactAttributeEvidence,
)

chunk_hash = "debug_chunk"
objs = []
for idx, item in enumerate(items):
    print(f"  [{idx}]", end=" ")
    try:
        etype = item.get("type", "").upper()
        conf = float(item.get("confidence", 0.5))
        if etype == "ENTITY":
            ev = EntityEvidence(chunk_hash, item.get("entity_name",""), item.get("entity_type","?"), conf, text[:200])
        elif etype == "ENTITY_ATTRIBUTE":
            ev = EntityAttributeEvidence(chunk_hash, item.get("entity_key",""), item.get("attr_key",""), item.get("attr_value",""), conf, text[:200])
        elif etype == "FACT":
            ev = FactEvidence(chunk_hash, item.get("subject_name",""), item.get("subject_type","?"), item.get("predicate",""), item.get("object_name",""), item.get("object_type","?"), conf, text[:200])
        elif etype == "FACT_ATTRIBUTE":
            ev = FactAttributeEvidence(chunk_hash, item.get("subject_key",""), item.get("predicate",""), item.get("object_key",""), item.get("attr_key",""), item.get("attr_value",""), conf, text[:200])
        else:
            print(f"SKIP unknown type: {etype}")
            continue
        objs.append(ev)
        print(f"OK → {ev.affected_key}")
    except Exception as e:
        print(f"ERROR: {e}")
        print(f"     item = {item}")
        traceback.print_exc()
print(f"Built {len(objs)} evidence objects")

  [0] OK → organization:社会保险基金经办机构
  [1] OK → organization:社会保险基金监督机构
  [2] OK → concept:社会保险基金
  [3] OK → organization:国家
  [4] OK → organization:用人单位
  [5] OK → concept:劳动者
  [6] OK → organization:劳动争议调解委员会
  [7] OK → organization:劳动争议仲裁委员会
  [8] OK → organization:人民法院
  [9] OK → organization:劳动行政部门
  [10] OK → organization:工会
  [11] OK → document:劳动法
  [12] OK → organization:劳动争议调解委员会|composition|职工代表、用人单位代表和工会代表
  [13] OK → organization:劳动争议调解委员会|chair|工会代表
  [14] OK → organization:劳动争议仲裁委员会|composition|劳动行政部门代表、同级工会代表、用人单位方面的代表
  [15] OK → organization:劳动争议仲裁委员会|chair|劳动行政部门代表
  [16] OK → organization:国家|DEVELOPS|concept:社会福利事业
  [17] OK → organization:国家|DEVELOPS|concept:公共福利设施
  [18] OK → organization:国家|SUPPORTS|concept:补充保险
Built 19 evidence objects


## 8. 写入 SQLite

In [43]:
from src.knowledge_graph.evidence_writer import write_evidence

result = write_evidence(chunk_hash, objs)
print(f"Wrote {result['count']} records")

from src.storage import doc_store
conn = doc_store._get_conn()
rows = conn.execute("SELECT evidence_type, COUNT(*) as cnt FROM evidence WHERE chunk_hash=? AND active=1 GROUP BY 1", (chunk_hash,)).fetchall()
for r in rows:
    print(f"  {r['evidence_type']}: {r['cnt']}")
conn.close()
print("Done.")

Wrote 19 records
  ENTITY: 98
  ENTITY_ATTRIBUTE: 46
  FACT: 66
  FACT_ATTRIBUTE: 14
Done.
